In [ ]:
import sys
from os.path import join
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.colors as mcolors
mm = 1.0/2.54/10

repo_root_path = join('..','..')

python_modules_paths = (
    join(repo_root_path, 'src', 'python'),
    'python'
)
for python_modules_path in python_modules_paths:
    if python_modules_path not in sys.path:
        sys.path.append(python_modules_path)

from foam.common import get_wpd_columns
from scm.scm import ilmenite, ilmenite_orig, Abad_SCR, Abad_SCRd, IlmenitePhase, parseReaction, concentration_from_molar_fraction
from labreactor.labreactor import LabReactor
from labreactor.labreactor_analytical import gas_yield_Leion
from labreactor.plotting import plot_labReactors_eff, plot_labReactors_exp_eff
from labreactor.plotting import plot_labReactor_eff

plt.style.use(os.path.join(repo_root_path, 'src', 'python', 'clc.mplstyle'))
# TODO: green color (third) is too close to blue (first)
# plt.style.use('tableau-colorblind10')
# ilmenite['act']['H2'].k(T=950+273)
from specie_properties.common import AW, MW

In [ ]:
def get_bed_heights(casename):
    # fo_name = 'graphUniform(start=(000),end=(00.160),fields=(p_rghalpha.solids),axis=y,nPoints=101,writeControl=timeStep,writeInterval=10)'
    fo_name = 'graphUniform(start=(000),end=(00.160),fields=(p_rghalpha.solids),axis=y,nPoints=101)'
    path_pp_fo = os.listdir(os.path.join(casename, 'postProcessing', fo_name))

    times = sorted(path_pp_fo, key=float)
    if '0' in times:
        times.remove('0')
    indicator_var = 'p_rgh'
    bed_heights = []
    for t in times:
        p = os.path.join(casename, 'postProcessing', fo_name, t, 'line.xy')
        df = pd.read_csv(p, skiprows=1, sep='\s+', names=['y', 'p_rgh', 'alpha.solids'])
        treshold = df[indicator_var].min() + 0.1*(df[indicator_var].max() - df[indicator_var].min())
        # print(casename, t)
        bed_height = df[df[indicator_var] < treshold].iloc[0]['y']
        bed_heights.append(bed_height)
    return [float(t) for t in times], bed_heights

## Sensitivity tests
### Time step

In [ ]:
casenames = [
    'labReactor_act_H2_3g_1050C_dt0.0000125',
    'labReactor_act_H2_3g_1050C',
    'labReactor_act_H2_3g_1050C_dt0.00005',
    'labReactor_act_H2_3g_1050C_dt0.0001',
    'labReactor_act_H2_3g_1050C_dt0.0002',
    ]
# NOTE: dt=0.000025 case is the original labReactor_act_H2_3g_1050C

def extract_dt(case_path):
    if 'dt' in case_path:
        return float(case_path.split('_')[-1].replace('dt', ''))
    else:
        return 0.000025

dt = extract_dt(casenames[0])
print(dt*1000)

In [ ]:
fig, ax = plt.subplots(figsize=(90*mm,67.5*mm))

for i, casename in enumerate(casenames):
    lr = LabReactor(casename)
    omega = lr.omega()
    dt = extract_dt(casename)
    ax.plot(omega.index, omega, label=fr'$\Delta t = {dt*1000}$ ms')

ax.legend()
ax.set(
    xlabel=r'$t$, s',
    ylabel=r'$\omega$, -'
)
fig.tight_layout()
fig.savefig('plots/labReactor_sensitivity_time_step_omega_vs_t.pdf')

In [ ]:
fig, ax = plt.subplots(figsize=(90*mm,67.5*mm))

for i, casename in enumerate(casenames):
    lr = LabReactor(casename)
    Tavgest = lr.temperature()
    dt = extract_dt(casename)
    ax.plot(Tavgest.index, Tavgest - 273.15, label=fr'$\Delta t = {dt*1000}$ ms')

ax.legend()
ax.set(
    xlabel=r'$t$, s',
    ylabel=r'$T$, °C',
    ylim = (None, ax.get_ylim()[1]+40)
)
ax.axhline(1050, color='gray',linestyle='--')
fig.tight_layout()
fig.savefig('plots/labReactor_sensitivity_time_step_T_vs_t.pdf')

In [ ]:
fig, ax = plt.subplots(figsize=(90*mm,67.5*mm))

for i, casename in enumerate(casenames):
    lr = LabReactor(casename)
    dt = extract_dt(casename)
    plot_labReactor_eff(ax, lr, 0.0, label=fr'$\Delta t = {dt*1000}$ ms')


ax.legend(loc='lower center')
ax.set(
    xlim = [1,0.96],
    xlabel = r'$\omega$, -',
    ylabel = r'$\gamma$, -',
)

fig.tight_layout()
fig.savefig('plots/labReactor_sensitivity_time_step_gamma_vs_omega.pdf')

### Density

In [ ]:
casenames = [
        'labReactor_act_CH4_rho2500',
        'labReactor_act_CH4_950C',
        'labReactor_act_CH4_rho3500',
        'labReactor_act_CH4_rho4000',
        'labReactor_act_CH4_rho4500',
    ]

In [ ]:
# for reference
fig, ax = plt.subplots(figsize=(90*mm,67.5*mm))

for casename in casenames:
    t, h = get_bed_heights(casename)
    lr = LabReactor(casename)
    ax.plot(t,1000*np.array(h), label=fr'$\rho = {lr.rhoOxidized} \, \mathrm{{kg}}/\mathrm{{m}}^3$')
ax.legend(ncols=2, loc='lower left', fontsize=7)
ax.set(
    xlabel=r'$t$, s',
    ylabel=r'$h_\mathrm{bed}$, mm',
    ylim = (-3, None)
)
fig.tight_layout()
fig.savefig('plots/labReactor_sensitivity_density_bed_height_vs_t.pdf')

In [ ]:
fig, ax = plt.subplots(figsize=(90*mm,67.5*mm))

for i, casename in enumerate(casenames):
    lr = LabReactor(casename)
    omega = lr.omega()
    ax.plot(omega.index, omega, label=fr'$\rho = {lr.rhoOxidized} \, \mathrm{{kg}}/\mathrm{{m}}^3$')

ax.legend()
ax.set(
    xlabel=r'$t$, s',
    ylabel=r'$\omega$, -',
    xlim = (0, 70)
)
fig.tight_layout()
fig.savefig('plots/labReactor_sensitivity_density_omega_vs_t.pdf')

In [ ]:
fig, ax = plt.subplots(figsize=(90*mm,67.5*mm))

for i, casename in enumerate(casenames):
    lr = LabReactor(casename)
    plot_labReactor_eff(ax, lr, 0.0, label=fr'$\rho = {lr.rhoOxidized} \, \mathrm{{kg}}/\mathrm{{m}}^3$')


ax.legend(loc='lower left', fontsize=7)
ax.set(
    xlim = [1,0.96],
    ylim = (None, 1),
    xlabel = r'$\omega$, -',
    ylabel = r'$\gamma$, -',
)
fig.tight_layout()
fig.savefig('plots/labReactor_sensitivity_density_gamma_vs_omega.pdf')

In [ ]:
fig, ax = plt.subplots(figsize=(90*mm,67.5*mm))

for i, casename in enumerate(casenames):
    lr = LabReactor(casename)
    Tavgest = lr.temperature()
    dt = extract_dt(casename)
    ax.plot(Tavgest.index, Tavgest - 273.15, label=fr'$\Delta t = {dt*1000}$ ms')

ax.legend()
ax.set(
    xlabel=r'$t$, s',
    ylabel=r'$T$, °C',
    # ylim = (None, ax.get_ylim()[1]+40)
)
ax.axhline(1050, color='gray',linestyle='--')
fig.tight_layout()
fig.savefig('plots/labReactor_sensitivity_density_T_vs_t.pdf')

### Mesh resolution

In [ ]:
casenames = [
        'labReactor_act_H2_3g_1050C_kN1', #  2 430 cells
        'labReactor_act_H2_3g_1050C',     # 19 440 cells
        'labReactor_act_H2_3g_1050C_kN3', # 46 170 cells
    ]

def getN(lr):
    return {
        1: 2430,
        2: 19440,
        3: 46170
    }[lr.kN]
# LabReactor(casenames[1]).__dict__

In [ ]:
fig, ax = plt.subplots(figsize=(90*mm, 67.5*mm))

for i, casename in enumerate(casenames):
    lr = LabReactor(casename)
    omega = lr.omega()
    ax.plot(omega.index, omega, label=fr'$N_\mathrm{{cells}} = {getN(lr)}$')

ax.legend()
ax.set(
    xlabel=r'$t$, s',
    ylabel=r'$\omega$, -',
    xlim = (0, 20)
)
fig.tight_layout()
fig.savefig('plots/labReactor_sensitivity_mesh_omega_vs_t.pdf')

In [ ]:
fig, ax = plt.subplots(figsize=(90*mm,67.5*mm))

for i, casename in enumerate(casenames):
    lr = LabReactor(casename)
    plot_labReactor_eff(ax, lr, 0.0, label=fr'$N_\mathrm{{cells}} = {getN(lr)}$')


ax.legend(loc='lower left')
ax.set(
    xlim = [1,0.96],
    xlabel = r'$\omega$, -',
    ylabel = r'$\gamma$, -',
)
fig.tight_layout()
fig.savefig('plots/labReactor_sensitivity_mesh_gamma_vs_omega.pdf')

In [ ]:
fig, ax = plt.subplots(figsize=(90*mm,67.5*mm))

for i, casename in enumerate(casenames):
    lr = LabReactor(casename)
    Tavgest = lr.temperature()
    dt = extract_dt(casename)
    ax.plot(Tavgest.index, Tavgest - 273.15, label=fr'$N_\mathrm{{cells}} = {getN(lr)}$')

ax.legend(loc='upper right')
ax.set(
    xlabel=r'$t$, s',
    ylabel=r'$T$, °C',
    ylim = (None, ax.get_ylim()[1]+5)
)
ax.axhline(1050, color='gray',linestyle='--')
fig.tight_layout()
fig.savefig('plots/labReactor_sensitivity_mesh_T_vs_t.pdf')